# Baseline Training

Trains the FP32 baseline CNN on CIFAR-10, with a fixed train/val/test split that will be reused by the PTQ, QAT, and binary quantisation notebooks.

In [ ]:
import tensorflow as tf
import numpy as np
import os

# For reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/tinyml-quant-security'
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)

In [ ]:
# To load CIFAR-10
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize to [0, 1] because epsilon values in adversarial attack (FGSM/PGD) use this range
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

# Fixed train/val split to control the result in every run
val_split = 0.1
num_train = len(x_train_full)
indices = np.arange(num_train)
rng = np.random.RandomState(SEED)
rng.shuffle(indices)

val_size = int(num_train * val_split)
val_idx, train_idx = indices[:val_size], indices[val_size:]

x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

# Save the split indices so every quantisation script uses the same split no.
np.savez(f'{PROJECT_DIR}/results/data_split.npz',
         train_idx=train_idx, val_idx=val_idx, seed=SEED)

In [ ]:
#### Baseline CNN architecture
"""
1. The architecture is small enough to be appropriate for TinyML
2. In this architecture, there is no augmentation layer because in a small model
like the inference, we don't need to have augmentation. This is because
Keras augmentation layers use TensorFlow operations such as StatelessRandomUniformV2 that are not supported
by the TFLite conversion toolchain. So, this can fail the TFLite converter.
"""
def build_baseline_model(num_classes=10):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),

        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_baseline_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# The augmentation has been done here
"""
Augmentation was implemented here in a tf.data preprocessing step
*** The augmentation is important because it can reduce overfitting, so the model
can learn more about the general features instead of memorising the pattern
"""

BATCH_SIZE = 64

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.resize_with_crop_or_pad(image, 36, 36)
    image = tf.image.random_crop(image, size=[32, 32, 3])
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(len(x_train), seed=SEED).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# The training

EPOCHS = 50 # Personally, I think it is a right no. for a small dataset like CIFAR-10
# yet it is not too small no. so that the model didn't learn properly

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=4, min_lr=1e-5),
    tf.keras.callbacks.ModelCheckpoint(
        f'{PROJECT_DIR}/models/baseline_best.keras',
        monitor='val_accuracy', save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# Saved as Keras for reloading later (maybe used in QAT/Binary)
model.save(f'{PROJECT_DIR}/models/baseline.keras')

# Saved the model for a further TFLite conversion
model.export(f'{PROJECT_DIR}/models/baseline_savedmodel')

In [ ]:
# Clean test evaluation
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Baseline FP32 -- Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")